In [47]:
import time
import tiktoken
import torch
import torch.nn as nn

In [48]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
            persistent=False
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        values = self.W_value(x)
        queries = self.W_query(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # context vector Shape: (b, num_heads, num_tokens, head_dim)
        context_vec = (attn_weights @ values) 

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = context_vec.transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec

In [49]:
torch.manual_seed(123)

# Define the tensor with 3 rows and 6 columns
inp = torch.tensor(
    [[0.43, 0.15, 0.89, 0.55, 0.87, 0.66],  
     [0.57, 0.85, 0.64, 0.22, 0.58, 0.33],  
     [0.77, 0.25, 0.10, 0.05, 0.80, 0.55]]
)

In [50]:
batch = torch.stack((inp, inp), dim=0)
print(batch.shape)

torch.Size([2, 3, 6])


In [51]:
batch_Size, context_length, d_in = batch.shape
d_out = 4

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)
print(context_vecs)

context_vecs.shape: torch.Size([2, 3, 4])
tensor([[[ 0.0903,  0.5205, -0.3216,  0.5317],
         [ 0.1708,  0.5399, -0.2854,  0.4249],
         [ 0.1818,  0.5396, -0.2860,  0.4306]],

        [[ 0.0903,  0.5205, -0.3216,  0.5317],
         [ 0.1708,  0.5399, -0.2854,  0.4249],
         [ 0.1818,  0.5396, -0.2860,  0.4306]]], grad_fn=<ViewBackward0>)


Entire GPT Architecture

In [52]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [54]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [55]:
model = GPTModel(GPT_CONFIG_124M)

Next Token Prediction 

Here wihtout training we are trying to predict the next token using the generate_simple_function()

In [56]:
tokenizer = tiktoken.get_encoding('gpt2')

In [70]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    # idx = idx.to(device)

    for i in range(max_new_tokens):
        # Crop current context if it exceeds the supported context size
        idx_cond = idx[:, -context_size:]

        print("idx : ", idx_cond)
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)     # batch, n_tokens, vocab_size
            print(i+1, "LLM pass  ",  "logits shape : ", list(logits.shape))
            # print(logits)

        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=-1)  # (batch, n_tokens+1)
      
    print(idx)
      

    return idx

In [71]:
start_context = "The next day is"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor)

encoded: [464, 1306, 1110, 318]
encoded_tensor.shape: tensor([[ 464, 1306, 1110,  318]])


In [72]:
model.eval() #A
#model = GPTModel(GPT_CONFIG_124M)
out = generate_text_simple(model=model, idx=encoded_tensor, max_new_tokens=6, context_size=GPT_CONFIG_124M['context_length'])

print()
print("Output:", out)
print("Output length:", len(out[0]))

idx :  tensor([[ 464, 1306, 1110,  318]])
1 LLM pass   logits shape :  [1, 4, 50257]
idx :  tensor([[ 464, 1306, 1110,  318, 3863]])
2 LLM pass   logits shape :  [1, 5, 50257]
idx :  tensor([[  464,  1306,  1110,   318,  3863, 38426]])
3 LLM pass   logits shape :  [1, 6, 50257]
idx :  tensor([[  464,  1306,  1110,   318,  3863, 38426, 12326]])
4 LLM pass   logits shape :  [1, 7, 50257]
idx :  tensor([[  464,  1306,  1110,   318,  3863, 38426, 12326, 44090]])
5 LLM pass   logits shape :  [1, 8, 50257]
idx :  tensor([[  464,  1306,  1110,   318,  3863, 38426, 12326, 44090, 15038]])
6 LLM pass   logits shape :  [1, 9, 50257]
tensor([[  464,  1306,  1110,   318,  3863, 38426, 12326, 44090, 15038, 12580]])

Output: tensor([[  464,  1306,  1110,   318,  3863, 38426, 12326, 44090, 15038, 12580]])
Output length: 10
